In [ ]:
import pandas as pd
import numpy as np
import joblib

 
# 1. LOAD CHAMPION AI MODEL AND THE NEW VIRTUAL CANDIDATE LIBRARY
 
print("🔄 Loading Perovskite Regressor Brain and Virtual Library...")
model = joblib.load('Perovskite_Regressor_Model.joblib')
df_candidates = pd.read_csv('Virtual_Candidate_Library.csv')

 
# 2. MATCH EXACT 24 DESCRIPTOR FEATURES USED DURING MODEL TRAINING
 
# These must match your model's exact shape and training sequence!
features = [
    'rA', 'rB', 'rX', 'chi_A', 'chi_B', 'chi_X', 'EA_A', 'EA_B', 'EA_X', 
    'tolerance_factor', 'octahedral_factor', 'a', 'b', 'c', 'volume', 'density', 
    'chi_diff_BX', 'r_ratio_BX', 'bond_len_BX', 'inv_bond_sq', 'lattice_strain'
]

# Extract feature matrix
X_candidates = df_candidates[features]

 
# 3. EXECUTE IN SILICO BANDGAP PREDICTIONS
 
print(f"🧠 Projecting quantum bandgaps onto {len(df_candidates)} hypothetical compounds...")
df_candidates['Predicted_BG'] = model.predict(X_candidates)

 
# 4. APPLY MULTI-STAGE "ELITE" PHOTVOLTAIC FILTERS (Physics Rules)
 
print("🔬 Running multi-stage filtering matrix for High-Efficiency & Stability...")

# Constraint 1: Thermodynamic Solar Sweet Spot (Single-junction limit window)
solar_filter = (df_candidates['Predicted_BG'] >= 1.1) & (df_candidates['Predicted_BG'] <= 1.7)

# Constraint 2: Goldschmidt Structural Stability Success Window
stability_filter = (df_candidates['tolerance_factor'] >= 0.80) & (df_candidates['tolerance_factor'] <= 1.05)

# Constraint 3: Strict Environmental Safety (100% Lead-Free)
# This safely checks the formula string to exclude any compound containing Lead ('Pb')
lead_free_filter = ~df_candidates['formula'].str.contains('Pb', case=True)

# Combine all filters into a single logical mask
elite_mask = solar_filter & stability_filter & lead_free_filter
df_elite = df_candidates[elite_mask].copy()

 
# 5. SORT BY DISCOVERY SCORE (Optimizing closest approach to ideal 1.34 eV)
 
# According to the Shockley-Queisser limit, 1.34 eV is the absolute thermodynamic 
# peak efficiency for a single-junction solar cell under standard AM1.5G sunlight.
df_elite['Deviation_From_Peak'] = abs(df_elite['Predicted_BG'] - 1.34)
df_elite = df_elite.sort_values(by='Deviation_From_Peak', ascending=True)

# Save the complete discovery pool to file
df_elite.to_csv("Elite_Solar_Discoveries.csv", index=False)

 
# 6. PRINT SUMMARY LOG OUTPUT
 
print("\n" + "="*80)
print(f"🌟 DISCOVERY COMPLETE: Found {len(df_elite)} Stable, Lead-Free Solar Candidates!")
print("="*80)

if len(df_elite) > 0:
    print("\n👑 TOP 10 HIGHEST-EFFICIENCY DISCOVERED MATERIALS (Sorted by Solar Viability):")
    print(df_elite[['formula', 'Predicted_BG', 'tolerance_factor', 'octahedral_factor']].head(10).to_string(index=False))
else:
    print("❌ No materials matched all three strict physical constraints simultaneously.")

print("\n" + "="*80)
print("💾 All elite candidate data successfully exported to: 'Elite_Solar_Discoveries.csv'")
print("="*80)

🔄 Loading Perovskite Regressor Brain and Virtual Library...
🧠 Projecting quantum bandgaps onto 211 hypothetical compounds...


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but GradientBoostingRegressor was fitted without feature names
  warnings.warn(


ValueError: X has 21 features, but GradientBoostingRegressor is expecting 24 features as input.